# 06A_ERA5_Feature_Extraction.ipynb
Extract ERA5-Land weather features from image_metadata.csv.

In [ ]:
!pip -q install earthengine-api pandas tqdm

import ee
import pandas as pd
from tqdm import tqdm
from google.colab import files

ee.Authenticate()
ee.Initialize(project='heatwave-flood-prediction')

print('Upload image_metadata.csv')
uploaded=files.upload()
fname=list(uploaded.keys())[0]
df=pd.read_csv(fname)

results=[]
skipped=[]

for _,row in tqdm(df.iterrows(),total=len(df)):
    event_id=row['event_id']
    d=ee.Date(str(row['image_date']))
    pt=ee.Geometry.Point([float(row['longitude']),float(row['latitude'])])

    col=(ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
         .filterBounds(pt)
         .filterDate(d.advance(-3,'day'),d.advance(4,'day'))
         .sort('system:time_start'))

    if col.size().getInfo()==0:
        skipped.append({'event_id':event_id,'reason':'No ERA5 image'})
        continue

    img=col.first()

    try:
        vals=img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=pt,
            scale=10000,
            maxPixels=1e9
        ).getInfo()

        t=vals.get('temperature_2m')
        dp=vals.get('dewpoint_temperature_2m')

        results.append({
            'event_id':event_id,
            'image_date':row['image_date'],
            'era5_temp_mean_c':None if t is None else t-273.15,
            'era5_dewpoint_mean_c':None if dp is None else dp-273.15,
            'era5_pressure_mean_pa':vals.get('surface_pressure'),
            'era5_u_wind_mean':vals.get('u_component_of_wind_10m'),
            'era5_v_wind_mean':vals.get('v_component_of_wind_10m'),
            'era5_precip_sum_mm':vals.get('total_precipitation_sum'),
            'era5_runoff_sum_mm':vals.get('runoff_sum')
        })
    except Exception as e:
        skipped.append({'event_id':event_id,'reason':str(e)})

pd.DataFrame(results).to_csv('era5_features.csv',index=False)
pd.DataFrame(skipped).to_csv('era5_skipped.csv',index=False)

files.download('era5_features.csv')
files.download('era5_skipped.csv')
print('Done')
